# Target acquisition

Take overview images, let the analysis find objects, then image the brightest few at high magnification.

It runs on the mock microscope. Run `python install.py` in this repository first.

## 1. Connect

In [ ]:
from zmart_controller import mic

mic.connect("mock", {"output_root": "images"})

## 2. Overview positions

An overview position is a place on the sample that you want to survey. Move the stage to each one, then record it with `get_xyz`. On a real microscope you would move the stage by hand or in the vendor software; the mock is moved from code so the notebook runs by itself. Add as many positions as you like.

In [ ]:
def where_am_i():
    xyz = mic.get_xyz()["content"]
    return [xyz["x"]["position"], xyz["y"]["position"], xyz["z"]["position"]]

overview_positions = []
for x, y in [(100, 100), (300, 100)]:
    mic.set_xyz(x, y, 0)
    overview_positions.append(where_am_i())

## 3. States

A *state* is the set of microscope settings (objective, laser power, exposure and so on) that you want to come back to later. We need two: the **overview** state, used to find objects, and the **target** state, used to look at them closely. Set the microscope up the way you want, then record it with `get_state`. Here the mock is set from code, with the 10x objective for the overview and the 40x for the targets.

In [ ]:
def current_state():
    return {"changeable": mic.get_state()["content"]["changeable"]}

mic.set_state({"changeable": {"objective": 1}})      # 10x: wide view
overview_state = current_state()

mic.set_state({"changeable": {"objective": 3}})      # 40x: close-up
target_state = current_state()

## 4. Analysis

The analysis engine runs the steps of a pipeline in the background. This pipeline finds nuclei and measures each one. Registering it only reads the pipeline file; nothing runs until an image is submitted. `ZMART_ANALYSIS` is the ZMART-analysis folder that `install.py` put in `work`, two folders up from this notebook.

In [ ]:
from zmart_analysis import Engine

ZMART_ANALYSIS = "../../work/ZMART-analysis"

engine = Engine()
engine.register(
    "find_objects",
    f"{ZMART_ANALYSIS}/workflows/object_analysis/pipelines/object_analysis_watershed.yaml",
    "analysis_output",
)

## 5. The loop

The loop has three parts. First it takes an image at every overview position and hands each one to the analysis, which works in the background while the stage keeps moving. Then it waits for the analysis to finish and chooses the best few objects from each table; here these are the brightest, but any column of the table will do. Finally it switches to the target state and images each chosen object at its stage position.

The analysis needs the pixel size and image orientation to turn an object's place in the image into a place on the stage. The pixel size comes from the microscope. Two values are written in by hand and are the mock's: the image size of 64 by 64 pixels, and the orientation `[[1, 0], [0, 1]]`. A real microscope has its own image size, and its own orientation, measured once during setup.

In [ ]:
import time

N_TARGETS = 5

# 1. Overview: take an image at each position and send it to the analysis.
mic.set_state(overview_state)
jobs = {}
for i, (x, y, z) in enumerate(overview_positions):
    mic.set_xyz(x, y, z)
    image = mic.acquire(position_label=f"overview{i}")["content"]
    pixel_um = mic.get_state()["content"]["observed"]["pixel_size"]["x"]
    jobs[i] = engine.submit("find_objects", {
        "image_path": image["files"][0],
        "tile_id": [f"overview{i}"],
        "tile_stage_xy_um": [x, y],
        "source_pixel_size_um": [pixel_um, pixel_um],
        "source_image_size_px": [64, 64],
        "image_to_stage": [[1, 0], [0, 1]],
    })

# 2. Wait for the analysis, then choose the brightest objects of each overview.
targets = []
for i, job in jobs.items():
    while (entry := engine.status("find_objects")[job])["state"] not in ("done", "failed"):
        time.sleep(0.2)
    assert entry["state"] == "done", entry.get("error")
    table = entry["output"]["object_analysis"]["objects"]["properties"]
    brightest = sorted(range(len(table["label"])), key=lambda k: -table["intensity_mean"][k])
    for k in brightest[:N_TARGETS]:
        targets.append((f"overview{i}_object{table['label'][k]}",
                        table["stage_x_um"][k], table["stage_y_um"][k], overview_positions[i][2]))

# 3. Targets: switch to the close-up state and visit each chosen object.
mic.set_state(target_state)
for label, x, y, z in targets:
    mic.set_xyz(x, y, z)
    mic.acquire(position_label=label)

When the loop is done, close both connections.

In [ ]:
engine.shutdown()
mic.disconnect()